# Polly 7B v2 — QLoRA Training on Colab T4

**Model**: Qwen/Qwen2.5-7B-Instruct → polly-scbe-7b-v2
**Data**: 2306 chat records from polly_combined_sft.jsonl
**Strategy**: QLoRA 4-bit, early stopping at best eval loss (~epoch 1.9)

Run all cells. Push to HuggingFace automatically.

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch transformers datasets peft accelerate bitsandbytes trl huggingface_hub
print('Dependencies installed.')

In [ ]:
# Cell 2: GPU check
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}, VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 3: HuggingFace auth
from huggingface_hub import login
import os

# Try Colab secrets first, then manual input
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    login(token=token)
    print('Logged in via Colab secrets')
except:
    token = input('Enter HF token: ')
    login(token=token)
    print('Logged in via manual token')

In [ ]:
# Cell 4: Download training data
from huggingface_hub import hf_hub_download
import json

path = hf_hub_download(
    'issdandavis/polly-training-data',
    'polly_combined_sft.jsonl',
    repo_type='dataset',
    local_dir='/content/data'
)

# Load and validate
records = []
with open(path, 'r', encoding='utf-8', errors='replace') as f:
    for line in f:
        try:
            rec = json.loads(line.strip())
            if 'messages' in rec and len(rec['messages']) >= 3:
                records.append(rec)
        except:
            continue

print(f'Loaded {len(records)} valid training records')
print(f'Sample roles: {[m["role"] for m in records[0]["messages"]]}')
print(f'System prompt: {records[0]["messages"][0]["content"][:100]}...')

In [ ]:
# Cell 5: Build dataset with train/eval split
from datasets import Dataset

ds = Dataset.from_list(records)
split = ds.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(split["train"])}, Eval: {len(split["test"])}')

# Preview
sample = split['train'][0]
for m in sample['messages']:
    print(f"  {m['role']}: {m['content'][:80]}...")

In [ ]:
# Cell 6: Load base model with QLoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
OUTPUT_DIR = '/content/polly-7b-v2'
HF_REPO = 'issdandavis/polly-scbe-7b-v2'

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading base model (4-bit)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

# LoRA config — r=32, targeting all linear layers
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('Model ready.')

In [ ]:
# Cell 7: Configure training — early stopping + best model saving
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback
import math

# Calculate steps for ~2 epochs (we know epoch 1.9 was best from v1)
n_train = len(split['train'])
effective_batch = 1 * 8  # per_device * grad_accum
steps_per_epoch = math.ceil(n_train / effective_batch)
print(f'Steps per epoch: {steps_per_epoch}')
print(f'Will train ~2 epochs = ~{steps_per_epoch * 2} steps')
print(f'Eval every {max(steps_per_epoch // 4, 25)} steps')

eval_steps = max(steps_per_epoch // 4, 25)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    
    # Train for 3 epochs max, but early stopping will likely cut at ~2
    num_train_epochs=3,
    
    # Batch size — conservative for T4 16GB with 7B model
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    
    # Learning rate
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    lr_scheduler_type='cosine',
    
    # Eval + early stopping
    eval_strategy='steps',
    eval_steps=eval_steps,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    
    # Saving
    save_strategy='steps',
    save_steps=eval_steps,
    save_total_limit=3,
    
    # Logging
    logging_steps=10,
    
    # Precision + memory
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    max_grad_norm=0.3,
    optim='paged_adamw_8bit',
    
    # Sequence length — 512 tokens for 7B is safe on T4
    max_length=512,
    packing=True,
    
    # No external reporting
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print(f'Trainer configured. Starting training...')

In [ ]:
# Cell 8: TRAIN
result = trainer.train()
print(f'\nTraining complete!')
print(f'Total steps: {result.global_step}')
print(f'Final train loss: {result.training_loss:.4f}')

# Eval on best checkpoint
eval_result = trainer.evaluate()
print(f'Best eval loss: {eval_result["eval_loss"]:.4f}')

In [ ]:
# Cell 9: Save adapter + push to HuggingFace
adapter_dir = f'{OUTPUT_DIR}/final_adapter'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Adapter saved to {adapter_dir}')

# Push adapter to HF
model.push_to_hub(HF_REPO, commit_message='Polly 7B v2 QLoRA adapter')
tokenizer.push_to_hub(HF_REPO, commit_message='Polly 7B v2 tokenizer')
print(f'Adapter pushed to {HF_REPO}')

In [ ]:
# Cell 10: Merge LoRA into base model and push merged version
print('Merging LoRA weights into base model...')
merged_model = model.merge_and_unload()

merged_repo = f'{HF_REPO}-merged'
merged_model.push_to_hub(merged_repo, commit_message='Polly 7B v2 merged model')
tokenizer.push_to_hub(merged_repo, commit_message='Polly 7B v2 tokenizer')
print(f'Merged model pushed to {merged_repo}')

In [ ]:
# Cell 11: Inference test — does Polly sound like Polly?
from transformers import pipeline

POLLY_SYS = ('You are Polly, the AI assistant for Aethermoor and the SCBE project. '
    'You know the 14-layer governance pipeline, Sacred Tongues (KO, AV, RU, CA, UM, DR), '
    'hyperbolic geometry for AI safety, and Mesh Foundry.')

questions = [
    'What are the Sacred Tongues and how do they work?',
    'Explain how the 14-layer pipeline keeps AI safe.',
    'What is Mesh Foundry and who is it for?',
    'How does hyperbolic geometry help with AI safety?',
    'Tell me about polyhedral friction scoring.',
    'Who is Izack Thorne?',
    'What does SCBE stand for?',
]

pipe = pipeline('text-generation', model=merged_model, tokenizer=tokenizer,
    max_new_tokens=200, temperature=0.7, top_p=0.9, do_sample=True,
    repetition_penalty=1.1)

for q in questions:
    msgs = [{'role': 'system', 'content': POLLY_SYS}, {'role': 'user', 'content': q}]
    result = pipe(msgs)
    answer = result[0]['generated_text'][-1]['content']
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    print(f'A: {answer}')

print(f'\n{"="*60}')
print('INFERENCE TEST COMPLETE')

In [ ]:
# Cell 12: Training summary
print('=' * 60)
print('POLLY 7B V2 TRAINING COMPLETE')
print('=' * 60)
print(f'Base model:  {MODEL_ID}')
print(f'Train size:  {len(split["train"])} records')
print(f'Eval size:   {len(split["test"])} records')
print(f'Steps:       {result.global_step}')
print(f'Train loss:  {result.training_loss:.4f}')
print(f'Eval loss:   {eval_result["eval_loss"]:.4f}')
print(f'Adapter:     {HF_REPO}')
print(f'Merged:      {HF_REPO}-merged')
print('=' * 60)
print('Models are on HuggingFace. Runtime can be disconnected.')